# 02 - Regresion Lineal: Simple y Multiple

---

## Contenido

1. Que es la Regresion Lineal
2. Regresion Lineal Simple (RLS)
3. Minimos Cuadrados Ordinarios (OLS)
4. Suposiciones del modelo
5. Pipeline completo con Scikit-Learn (RLS)
6. Regresion Lineal Multiple (RLM)
7. Ejemplo con el dataset Iris

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid')
sns.set()

## 1. Que es la Regresion Lineal

La **Regresion Lineal (RL)** es el modelo mas simple para entender los problemas de regresion:

$$y = f(\mathbf{x}, \boldsymbol{\theta})$$

$$\mathbf{x} \in \mathbb{R}^N,\ \boldsymbol{\theta} \in \mathbb{R}^P,\ y \in \mathbb{R}$$

Este tipo de modelos son **muy populares** debido a que:

- Son *baratos* y *rapidos* de estimar
- Son **interpretables**

El caso base consiste en ajustar una linea (o hiperplano) a un conjunto de puntos para modelar la relacion entre variables independientes y una variable dependiente.

---

## 2. Regresion Lineal Simple (RLS)

Una Regresion Lineal Simple es una linea en 2D que es **ajustada** a datos y tiene la siguiente forma:

$$y = \beta_0 + \beta_1 x_1$$

donde:
- $y$ es la variable dependiente (lo que queremos predecir)
- $x_1$ es la variable independiente (el predictor)
- $\beta_0$ es la **ordenada al origen** (intercepto): el valor de $y$ cuando $x_1 = 0$
- $\beta_1$ es la **pendiente**: cuanto cambia $y$ por cada unidad de cambio en $x_1$

Esta funcion tambien se puede escribir en terminos del **valor esperado** de $Y$:

$$E(Y \mid X = x) = \beta_0 + \beta_1 x$$

---

## 3. Minimos Cuadrados Ordinarios (OLS)

### Residuos y la Suma de sus Cuadrados

El **ajuste optimo** se puede determinar de varias formas, pero la manera mas comun (y la utilizada por Scikit-Learn) es mediante el **metodo de OLS**, el cual minimiza una cantidad denominada *suma de cuadrados de los residuos* (RSS):

$$\text{Minimizar } RSS(\hat{\beta}_0, \hat{\beta}_1) = \sum_{i=1}^{n} \left[y_i - (\hat{\beta}_0 + \hat{\beta}_1 x_i)\right]^2$$

Donde:
- $y_i$ es el valor real de la observacion $i$
- $\hat{y}_i = \hat{\beta}_0 + \hat{\beta}_1 x_i$ es el valor predicho
- $e_i = y_i - \hat{y}_i$ es el **residuo** de la observacion $i$

### Solucion Analitica

Derivando e igualando a cero, se obtiene la solucion cerrada:

$$\hat{\boldsymbol{\beta}} = (X^T X)^{-1} X^T y$$

donde $X$ es la matriz de diseno (con una columna de unos para el intercepto).

### Ejemplo: Calculo manual de OLS

Generamos datos sinteticos que provienen de una linea con $\beta_0 = 2$ y $\beta_1 = 5$:

In [ ]:
from numpy.linalg import inv

# Parametros reales
b0_real = 2
b1_real = 5

# Generar datos sinteticos
rng = np.random.RandomState(1)
n_samples = 100

x = 5 * rng.rand(n_samples)
y = (b0_real + b1_real * x) + rng.randn(n_samples)  # Distribucion Normal como ruido

# Construir la matriz de diseno X con columna de unos
X_design = np.column_stack([np.ones(n_samples), x])

# Solucion analitica: beta = (X^T X)^{-1} X^T y
beta_hat = inv(X_design.T @ X_design) @ X_design.T @ y

print(f'Parametros reales:    beta_0 = {b0_real}, beta_1 = {b1_real}')
print(f'Parametros estimados: beta_0 = {beta_hat[0]:.4f}, beta_1 = {beta_hat[1]:.4f}')

In [ ]:
# Visualizar el ajuste por OLS
x_line = np.linspace(0, 5, 100)
y_hat = beta_hat[0] + beta_hat[1] * x_line

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(x, y, color='steelblue', alpha=0.6, label='Datos observados')
ax.plot(x_line, y_hat, color='red', linewidth=2,
        label=f'OLS: y = {beta_hat[0]:.2f} + {beta_hat[1]:.2f}x')
ax.set_xlabel('x', fontsize=12)
ax.set_ylabel('y', fontsize=12)
ax.set_title('Regresion Lineal Simple - Solucion por OLS', fontsize=13)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

---

## 4. Suposiciones del Modelo

La Regresion Lineal tiene varias suposiciones:

| Suposicion | Descripcion |
|---|---|
| **Linealidad** | La relacion entre X y $E[Y]$ es lineal |
| **Variables estandarizadas** | Se recomienda estandarizar: $x' = \frac{x - \bar{x}}{\sigma}$ |
| **Homocedasticidad** | La varianza del residuo es constante para cualquier valor de X |
| **Independencia** | Las observaciones son independientes entre si |
| **Sin colinealidad** | Las variables independientes no deben estar altamente correlacionadas |
| **Normalidad de residuos** | Los residuos siguen una distribucion normal |

### Verificacion de normalidad de residuos

In [ ]:
# Calcular los residuos del ejemplo anterior
y_predicted = beta_hat[0] + beta_hat[1] * x
residuals = y - y_predicted

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histograma de residuos
axes[0].hist(residuals, bins=15, edgecolor='black', color='steelblue', alpha=0.7)
axes[0].axvline(x=0, color='red', linestyle='--', linewidth=1.5)
axes[0].set_title('Distribucion de los Residuos', fontsize=12)
axes[0].set_xlabel('Residuo', fontsize=11)
axes[0].set_ylabel('Frecuencia', fontsize=11)

# Residuos vs valores predichos
axes[1].scatter(y_predicted, residuals, color='steelblue', alpha=0.6)
axes[1].axhline(y=0, color='red', linestyle='--', linewidth=1.5)
axes[1].set_title('Residuos vs Valores Predichos', fontsize=12)
axes[1].set_xlabel('Valor Predicho', fontsize=11)
axes[1].set_ylabel('Residuo', fontsize=11)

plt.tight_layout()
plt.show()

print(f'Media de los residuos: {np.mean(residuals):.6f}')
print(f'Desviacion estandar de los residuos: {np.std(residuals):.4f}')

---

## 5. Pipeline completo con Scikit-Learn (RLS)

Ahora veamos como implementar exactamente lo mismo pero usando Scikit-Learn. Seguimos los 5 pasos del pipeline clasico.

In [ ]:
# Paso 1: Importar la clase del modelo
from sklearn.linear_model import LinearRegression

In [ ]:
# Paso 2: Instanciar el modelo
model = LinearRegression(fit_intercept=True)
print(model)

In [ ]:
# Paso 3: Preparar los datos
# Scikit-Learn requiere que X sea 2D: (n_samples, n_features)
X = x[:, np.newaxis]  # Convierte de (100,) a (100, 1)
print(f'Forma de X: {X.shape}')
print(f'Forma de y: {y.shape}')

In [ ]:
# Paso 4: Ajustar el modelo
model.fit(X, y)

# Parametros aprendidos (con guion bajo al final por convencion)
print(f'Coeficiente (beta_1): {model.coef_[0]:.4f}')
print(f'Intercepto  (beta_0): {model.intercept_:.4f}')

Observa que los coeficientes obtenidos con Scikit-Learn son identicos a los calculados manualmente con OLS. Esto es porque Scikit-Learn utiliza internamente el mismo metodo.

In [ ]:
# Paso 5: Predecir sobre nuevos datos
xfit = np.linspace(-1, 6, 100)
Xfit = xfit[:, np.newaxis]
y_hat = model.predict(Xfit)

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(x, y, color='red', alpha=0.6, label='Datos de entrenamiento')
ax.plot(xfit, y_hat, color='blue', linewidth=2, label='Prediccion del modelo')
ax.set_xlabel('x', fontsize=12)
ax.set_ylabel('y', fontsize=12)
ax.set_title('RLS con Scikit-Learn', fontsize=13)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

### Inferencia estadistica con Statsmodels

Scikit-Learn no proporciona herramientas para obtener conclusiones estadisticas (p-valores, intervalos de confianza). Para eso, se usa la libreria `statsmodels`:

In [ ]:
import statsmodels.api as sm

# Agregar columna de constante para el intercepto
X_with_const = sm.add_constant(X)

# Ajustar modelo OLS con statsmodels
results = sm.OLS(y, X_with_const).fit()
print(results.summary())

---

## 6. Regresion Lineal Multiple (RLM)

La RLM extiende la RLS a multiples variables independientes:

$$y = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + \cdots + \beta_p x_p$$

donde $\mathbf{x} = (x_1, x_2, \ldots, x_p)$ es el vector de caracteristicas.

Dado que los humanos solo podemos percibir visualmente 3 dimensiones, es dificil visualizar una RLM. Sin embargo, podemos observar los coeficientes para cada variable e interpretar su contribucion.

In [ ]:
# Ejemplo con datos sinteticos de 3 dimensiones
rng = np.random.RandomState(1)
X_multi = 10 * rng.rand(100, 3)

# Parametros reales:     b1=1.5,  b2=-2.0, b3=1.0
y_multi = 0.5 + np.dot(X_multi, [1.5, -2., 1.])

# Visualizar relaciones con pairplot
Y_col = y_multi[:, np.newaxis]
data_multi = np.concatenate((X_multi, Y_col), axis=1)
df_multi = pd.DataFrame(data_multi, columns=['X1', 'X2', 'X3', 'Y'])

sns.pairplot(df_multi, kind='reg', corner=True, height=2.5,
             plot_kws={'scatter_kws': {'alpha': 0.5}});
plt.suptitle('Relaciones entre variables (datos sinteticos)', y=1.02, fontsize=13);

In [ ]:
# Ajustar RLM con Scikit-Learn
model_multi = LinearRegression(fit_intercept=True)
model_multi.fit(X_multi, y_multi)

print(f'Intercepto estimado: {model_multi.intercept_:.4f} (real: 0.5)')
print(f'Coeficientes estimados: {model_multi.coef_}')
print(f'Coeficientes reales:    [1.5, -2.0, 1.0]')

---

## 7. Ejemplo con el dataset Iris

Apliquemos la RLM al dataset Iris. El objetivo sera predecir `petal_width` usando las otras 3 variables numericas.

In [ ]:
# Cargar datos
iris = sns.load_dataset('iris')

# Definir X (3 caracteristicas) e y (petal_width)
features = ['sepal_length', 'sepal_width', 'petal_length']
target = 'petal_width'

X_iris = iris[features].values
y_iris = iris[target].values

print(f'X shape: {X_iris.shape}')
print(f'y shape: {y_iris.shape}')

In [ ]:
from sklearn.model_selection import train_test_split

# Dividir en entrenamiento (70%) y prueba (30%)
X_train, X_test, y_train, y_test = train_test_split(
    X_iris, y_iris, test_size=0.30, random_state=42
)

print(f'Entrenamiento: X={X_train.shape}, y={y_train.shape}')
print(f'Prueba:        X={X_test.shape}, y={y_test.shape}')

In [ ]:
# Entrenar el modelo
model_iris = LinearRegression()
model_iris.fit(X_train, y_train)

print('Coeficientes del modelo:')
for feat, coef in zip(features, model_iris.coef_):
    print(f'  {feat}: {coef:.4f}')
print(f'  Intercepto: {model_iris.intercept_:.4f}')

In [ ]:
# Predecir sobre el conjunto de prueba
y_pred = model_iris.predict(X_test)

# Visualizar predicciones vs valores reales
fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(y_test, y_pred, color='steelblue', alpha=0.7, edgecolors='black', linewidth=0.5)
ax.plot([0, 2.5], [0, 2.5], 'r--', linewidth=1.5, label='Linea ideal (y = y_pred)')
ax.set_xlabel('Valor Real (petal_width)', fontsize=12)
ax.set_ylabel('Valor Predicho (petal_width)', fontsize=12)
ax.set_title('RLM sobre Iris: Valores Reales vs Predichos', fontsize=13)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import r2_score, mean_squared_error

r2 = r2_score(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)

print(f'R^2:  {r2:.4f}')
print(f'MSE:  {mse:.4f}')
print(f'RMSE: {rmse:.4f}')
print()
print(f'Interpretacion: El modelo explica el {r2*100:.1f}% de la varianza de petal_width.')

---

## Resumen

En este notebook aprendimos:

1. La **RLS** modela la relacion entre una variable independiente y una dependiente: $y = \beta_0 + \beta_1 x$
2. El metodo de **OLS** minimiza la suma de cuadrados de los residuos para encontrar los parametros optimos.
3. El modelo tiene **suposiciones** que deben verificarse (linealidad, homocedasticidad, normalidad de residuos, etc.).
4. **Scikit-Learn** implementa la RL con la misma API uniforme: `fit()`, `predict()`, `score()`.
5. La **RLM** extiende la RLS a multiples variables: $y = \beta_0 + \beta_1 x_1 + \cdots + \beta_p x_p$.
6. Aplicamos RLM al dataset Iris para predecir `petal_width`.

En el siguiente notebook veremos la **Regresion Polinomial con Funciones Base**.